In [1]:
import sqlite3
import json
import os
from pathlib import Path

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

config_path = Path("config.json")
with config_path.open(encoding="utf-8") as config_file:
    config = json.load(config_file)

os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

DB_PATH = "kartify.db"

In [15]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    SELECT order_id, customer_id, product_description, order_status, tracking_number, shipping_partner
    FROM orders
    LIMIT 10
""")

rows = cursor.fetchall()
conn.close()

rows


[('O75592',
  'C1014',
  'Bluetooth Speaker',
  'Confirmed',
  'TRK947129',
  'Delhivery'),
 ('O40327', 'C1010', 'Office Chair', 'Confirmed', 'TRK552659', 'UPS'),
 ('O91604', 'C1019', 'Bluetooth Speaker', 'Confirmed', 'TRK123530', 'DHL'),
 ('O69316', 'C1006', 'Wireless Mouse', 'Confirmed', 'TRK495949', 'Delhivery'),
 ('O74260', 'C1008', 'Wireless Mouse', 'Confirmed', 'TRK901849', 'BlueDart'),
 ('O55611', 'C1011', 'Yoga Mat', 'Confirmed', 'TRK699158', 'FedEx'),
 ('O81247', 'C1016', 'LED Monitor', 'Confirmed', 'TRK130207', 'BlueDart'),
 ('O63257', 'C1017', 'Electric Kettle', 'Confirmed', 'TRK745816', 'FedEx'),
 ('O34866', 'C1015', 'Yoga Mat', 'Confirmed', 'TRK435071', 'BlueDart'),
 ('O17780', 'C1008', 'Yoga Mat', 'Confirmed', 'TRK478293', 'Delhivery')]

In [3]:
@tool
def fetch_order_details(order_id: str) -> str:
    """Fetch order details for a given Kartify order ID."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            order_id,
            customer_id,
            product_description,
            order_date,
            payment_status,
            order_status,
            price_paid,
            return_days_allowed,
            return_conditions,
            replacement_days_allowed,
            product_warranty,
            tracking_number,
            shipping_partner,
            dispatch_date,
            expected_delivery,
            actual_delivery
        FROM orders
        WHERE order_id = ?
    """, (order_id,))

    row = cursor.fetchone()
    conn.close()

    if not row:
        return f"No order found for order ID {order_id}."

    (
        order_id,
        customer_id,
        product_description,
        order_date,
        payment_status,
        order_status,
        price_paid,
        return_days_allowed,
        return_conditions,
        replacement_days_allowed,
        product_warranty,
        tracking_number,
        shipping_partner,
        dispatch_date,
        expected_delivery,
        actual_delivery,
    ) = row

    return (
        f"Order ID: {order_id}. "
        f"Customer ID: {customer_id}. "
        f"Product: {product_description}. "
        f"Order date: {order_date}. "
        f"Payment status: {payment_status}. "
        f"Order status: {order_status}. "
        f"Price paid: {price_paid}. "
        f"Return window: {return_days_allowed} days. "
        f"Return conditions: {return_conditions}. "
        f"Replacement window: {replacement_days_allowed} days. "
        f"Product warranty: {product_warranty} months. "
        f"Tracking number: {tracking_number}. "
        f"Shipping partner: {shipping_partner}. "
        f"Dispatch date: {dispatch_date}. "
        f"Expected delivery: {expected_delivery}. "
        f"Actual delivery: {actual_delivery or 'Not delivered yet'}."
    )

In [4]:
@tool
def random_number():
    '''Returns a random number between 1 and 100.'''
    import random
    return random.randint(1, 100)



In [5]:
#print(fetch_order_details("O40327"))

'''
    Here we call the LLM directly so it can answer the user's question.
    The base LLM does not have direct access to the database or to fetch_order_details.
    If it gives a specific order answer here, that answer is not grounded in the SQLite data.
 
'''
question = "Where is my order O40327?"

response = llm.invoke([
    HumanMessage(content=question)
])

print(response.content)


# testing the tool directly
#fetch_order_details.invoke({"order_id": "O40327"})




I'm sorry, but I don't have access to order tracking systems or personal data. To find the status of your order O40327, please check the confirmation email you received or visit the website of the retailer where you placed the order. You can usually find tracking information in your account or by contacting customer service directly.


In [6]:
@tool
def random_letter() -> str:
    '''Returns a random lowercase letter. Use this when the user asks for a random character.'''
    import random
    import string

    return random.choice(string.ascii_lowercase)


In [7]:
# this line binds the tool to the LLM, allowing it to call the tool when needed
# llm_with_tools is a new instance of the LLM that has the fetch_order_details tool bound to it. This means that when llm_with_tools is invoked, it will be able to call fetch_order_details when it determines that it needs to in order to answer the user's question.
tools = [fetch_order_details, random_number, random_letter]
llm_with_tools = llm.bind_tools(tools)

user_question = "give me details about order O40327 and a random number between 1 and 100, and a random letter"

# this line simulates a user asking a question that requires the tool to be called. The LLM will recognize that it needs to call the tool, and will do so automatically.
tool_request = llm_with_tools.invoke([
    HumanMessage(content=user_question)
])

# Create a lookup dictionary so we can execute a tool by the name chosen by the LLM.
# Example:
# {
#     "fetch_order_details": fetch_order_details,
#     "random_number": random_number
# }
tools_by_name = {tool.name: tool for tool in tools}

# This list will store the results of each tool execution.
# Later, we send these results back to the LLM as ToolMessage objects.
tool_messages = []

if 1 == 0:
    exit

# tool_request.tool_calls contains the tools that the LLM decided it needs.
# Important: the LLM has not executed the tools yet. It only requested them.
for call in tool_request.tool_calls:

    # Get the tool object that matches the name requested by the LLM.
    # Example: if call["name"] is "fetch_order_details", this selects that tool.
    selected_tool = tools_by_name.get(call["name"])

    if selected_tool is None:
        raise ValueError(f"Unknown tool requested: {call['name']}")
    
    # Execute the selected tool with the arguments generated by the LLM.
    # Example: {"order_id": "O40327"}
    result = selected_tool.invoke(call["args"])

    # Wrap the tool result in a ToolMessage.
    # tool_call_id links this result back to the exact tool call requested by the LLM.
    tool_messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )


# The tool requested by the LLM is stored in tool_request.tool_calls.
# This is a list of every tool the LLM decided it needs to call.
#tool_call = tool_request.tool_calls

#for tool_call in tool_request.tool_calls:
#    print(f"Tool name: {call['name']}\n" + 40 * "-")
#    for key, value in call.items():
#        print(f"{key}: {value}")

    #tool_result = fetch_order_details.invoke(tool_call["args"])
        


    

In [8]:
# this line binds the tool to the LLM, allowing it to call the tool when needed
# llm_with_tools is a new instance of the LLM that has the fetch_order_details tool bound to it. This means that when llm_with_tools is invoked, it will be able to call fetch_order_details when it determines that it needs to in order to answer the user's question.

tools = [fetch_order_details, random_number]
llm_with_tools = llm.bind_tools(tools)

user_question = "Where is my order O40327?"

# this line simulates a user asking a question that requires the tool to be called. 
# The LLM will recognize that it needs to call the tool, and will do so automatically.
tool_request = llm_with_tools.invoke([
    HumanMessage(content=user_question)
])

# The tool requested by the LLM is stored in tool_request.tool_calls.
# This is a list of every tool the LLM decided it needs to call.
# In this case, there should be only one requested tool: fetch_order_details.
tool_call = tool_request.tool_calls[0]

# Get the tool result by calling fetch_order_details.invoke with the arguments generated by the LLM.
# In this case, the argument should be the order_id "O40327".
tool_result = fetch_order_details.invoke(tool_call["args"])

#tool_result

# Ask the LLM for the final answer after giving it the tool result.
# The LLM can use the tool output to generate a more complete user-facing response.
final_response = llm_with_tools.invoke([
    HumanMessage(content="Where is my order O40327?"),
    tool_request,
    ToolMessage(
        content=tool_result,
        tool_call_id=tool_call["id"]
    )
])

print(final_response.content)

Your order O40327 is for an Office Chair. Here are the details:

- **Order Date:** 02-07
- **Payment Status:** Paid
- **Order Status:** Confirmed
- **Price Paid:** $133.86
- **Return Window:** 0 days (Not eligible for return)
- **Replacement Window:** 15 days
- **Product Warranty:** 90 months
- **Tracking Number:** TRK552659
- **Shipping Partner:** UPS
- **Dispatch Date:** 05-07
- **Expected Delivery:** 12-07
- **Actual Delivery:** 14-07

Your order was delivered on 14-07. If you need further assistance, feel free to ask!


In [12]:
from langchain_core.tools import tool

@tool
def suma(a, b):
    '''add two numbers together'''
    return a + b

@tool
def resta(a, b):
    '''substract two numbers'''
    return a - b

@tool
def multiplicacion(a, b):
    '''multiply two numbers'''
    return a * b


llm_operations = ChatOpenAI(model="gpt-4o-mini", temperature=0)

functions = [suma, resta, multiplicacion]

user_question = "What is the sum of 5 and 10, the difference of 27 and 5, and the product of 3 and 4?"

# tool binding for the operations tools
llm_with_operations = llm_operations.bind_tools(functions)


# llm call to process the user question and determine which tools to invoke
# this is going to return a ToolRequest object that contains the tools 
# that the LLM decided it needs to call in order to answer the user's question.
tool_request = llm_with_operations.invoke([
    HumanMessage(content=user_question)
])

# Dictionary that maps each function name to the actual tool object so it can be invoked later.
function_names = {
    "suma": suma,
    "resta": resta,
    "multiplicacion": multiplicacion
}

for tool_call in tool_request.tool_calls:
    function_name = function_names.get(tool_call["name"])
    args_function = tool_call["args"]
    function_result = function_name.invoke(args_function)

    print(f"Function name: {tool_call["name"]}")
    print(f"Arguments: {args_function}")
    print(f"Function result: {function_result} \n")



Function name: suma
Arguments: {'a': 5, 'b': 10}
Function result: 15 

Function name: resta
Arguments: {'a': 27, 'b': 5}
Function result: 22 

Function name: multiplicacion
Arguments: {'a': 3, 'b': 4}
Function result: 12 

